# 01 - Entrainement ModCloth Fit Model V0

Objectif : entrainer le modele TensorFlow/Keras tabulaire de prediction du fit (`small`, `fit`, `large`) a partir du dataset ModCloth.

Ce notebook est limite au pipeline ModCloth V0 : recuperation Kaggle, inspection du dataset, appel du script `src.training.train_fit_model`, verification des artefacts, puis copie vers Google Drive.

Il ne travaille pas sur le CNN Fashion Product Images ni sur Polyvore.

## 1. Montage Google Drive

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


## 2. Clonage ou mise a jour du repo GitHub

Renseigne `REPO_URL` avec l'URL GitHub du projet, ou cree un secret Colab `FIT_OUTFIT_REPO_URL` contenant cette URL.

In [9]:
import os
from pathlib import Path
from google.colab import userdata

DEFAULT_REPO_URL = 'https://github.com/MilFhey/fit-outfit-advisor.git'
REPO_URL = DEFAULT_REPO_URL
REPO_DIR = Path('/content/fit-outfit-advisor')
BRANCH = 'main'


if REPO_DIR.exists():
    print(f'Repo deja present, mise a jour : {REPO_DIR}')
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull --ff-only origin {BRANCH}
else:
    print(f'Clonage du repo : {REPO_URL}')
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f'Repertoire courant : {Path.cwd()}')

Clonage du repo : https://github.com/MilFhey/fit-outfit-advisor.git
Cloning into '/content/fit-outfit-advisor'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 61 (delta 9), reused 61 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (61/61), 34.18 KiB | 11.39 MiB/s, done.
Resolving deltas: 100% (9/9), done.
Repertoire courant : /content/fit-outfit-advisor


## 3. Installation des dependances

In [10]:
!python -m pip install --upgrade pip
!python -m pip install -r requirements.txt
!python -m pip install kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 58.0 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


## 4. Creation des dossiers temporaires dans `/content`

In [11]:
CONTENT_ROOT = Path('/content/fit-outfit-runtime')
KAGGLE_DOWNLOAD_DIR = CONTENT_ROOT / 'kaggle_downloads'
CONTENT_DATA_DIR = CONTENT_ROOT / 'data'
CONTENT_ARTIFACT_DIR = CONTENT_ROOT / 'artifacts'

for directory in [CONTENT_ROOT, KAGGLE_DOWNLOAD_DIR, CONTENT_DATA_DIR, CONTENT_ARTIFACT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
    print(directory)

/content/fit-outfit-runtime
/content/fit-outfit-runtime/kaggle_downloads
/content/fit-outfit-runtime/data
/content/fit-outfit-runtime/artifacts


## 5. Chargement securise du secret Colab `KAGGLE_API_TOKEN`

Le secret peut contenir soit le JSON Kaggle complet (`{"username":"...","key":"..."}`), soit le format `username:key`. La valeur n'est jamais affichee.

In [15]:
import os
from google.colab import userdata

# Récupère le token moderne Kaggle depuis Colab Secrets.
# Le secret doit être nommé exactement : KAGGLE_API_TOKEN
try:
    kaggle_token = userdata.get("KAGGLE_API")
except Exception as exc:
    raise ValueError(
        "Impossible de lire le secret Colab KAGGLE_API. "
        "Ajoute-le dans le panneau Secrets de Colab."
    ) from exc

if not kaggle_token or not kaggle_token.startswith("KGAT_"):
    raise ValueError(
        "Le secret KAGGLE_API est absent ou invalide. "
        "Il doit contenir uniquement ton token Kaggle moderne commençant par KGAT_."
    )

# Rend le token disponible pour la commande `kaggle`
os.environ["KAGGLE_API"] = kaggle_token

print("✅ Token Kaggle chargé depuis Colab Secrets.")

✅ Token Kaggle chargé depuis Colab Secrets.


In [16]:
!pip install -q -U kaggle
!kaggle datasets list -s modcloth

Authentication required to call the Kaggle API.

First, you will need a Kaggle account. You can sign up at
  https://www.kaggle.com/account/login

Recommended: log in with OAuth via a web-based authorization flow.
No token to manage; credentials are cached locally for you.
    kaggle auth login

If you'd rather not use OAuth, generate an API token at
  https://www.kaggle.com/settings/api  (click "Generate New Token" under "API")
and supply it to the CLI in one of these ways:

  Option A: Environment variable
    export KAGGLE_API_TOKEN=xxxxxxxxxxxxxx  # token copied from the settings UI

  Option B: API token file
    Save the token to ~/.kaggle/access_token


## 6. Telechargement du dataset ModCloth via Kaggle API

In [17]:
KAGGLE_DATASET = 'rmisra/clothing-fit-dataset-for-size-recommendation'

!kaggle datasets download -d {KAGGLE_DATASET} -p {KAGGLE_DOWNLOAD_DIR} --unzip

downloaded_files = sorted(path for path in KAGGLE_DOWNLOAD_DIR.rglob('*') if path.is_file())
if not downloaded_files:
    raise FileNotFoundError('Aucun fichier telecharge depuis Kaggle. Verifie le token et le slug dataset.')

for path in downloaded_files:
    print(path)

Dataset URL: https://www.kaggle.com/datasets/rmisra/clothing-fit-dataset-for-size-recommendation
License(s): Attribution 4.0 International (CC BY 4.0)
100% 39.7M/39.7M [00:00<00:00, 57.4MB/s]

/content/fit-outfit-runtime/kaggle_downloads/modcloth_final_data.json
/content/fit-outfit-runtime/kaggle_downloads/renttherunway_final_data.json


## 7. Detection et affichage du fichier CSV ModCloth

Si Kaggle fournit ModCloth en JSON/JSONL, le notebook le convertit en CSV temporaire pour inspection et pour l'appel du script.

In [18]:
import pandas as pd

csv_files = sorted(KAGGLE_DOWNLOAD_DIR.rglob('*.csv'))
modcloth_csv_files = [path for path in csv_files if 'modcloth' in path.name.lower()]

if modcloth_csv_files:
    DATASET_PATH = modcloth_csv_files[0]
    print(f'CSV ModCloth detecte : {DATASET_PATH}')
elif csv_files:
    DATASET_PATH = csv_files[0]
    print(f'CSV detecte : {DATASET_PATH}')
else:
    json_files = sorted(
        [*KAGGLE_DOWNLOAD_DIR.rglob('*.json'), *KAGGLE_DOWNLOAD_DIR.rglob('*.jsonl')]
    )
    modcloth_json_files = [path for path in json_files if 'modcloth' in path.name.lower()]
    if not modcloth_json_files:
        raise FileNotFoundError('Aucun CSV ni JSON ModCloth detecte dans le telechargement Kaggle.')

    source_json = modcloth_json_files[0]
    print(f'JSON ModCloth detecte : {source_json}')
    df_json = pd.read_json(source_json, lines=True)
    DATASET_PATH = CONTENT_DATA_DIR / 'modcloth_final_data.csv'
    df_json.to_csv(DATASET_PATH, index=False)
    print(f'CSV temporaire genere : {DATASET_PATH}')

print(f'DATASET_PATH = {DATASET_PATH}')

JSON ModCloth detecte : /content/fit-outfit-runtime/kaggle_downloads/modcloth_final_data.json
CSV temporaire genere : /content/fit-outfit-runtime/data/modcloth_final_data.csv
DATASET_PATH = /content/fit-outfit-runtime/data/modcloth_final_data.csv


## 8. Inspection du dataset

In [19]:
df = pd.read_csv(DATASET_PATH)

print('df.shape =', df.shape)
print('\ndf.columns =')
print(list(df.columns))

display(df.head())

missing_values = df.isna().sum().sort_values(ascending=False)
print('\nValeurs manquantes par colonne :')
display(missing_values.to_frame('missing_count'))

df.shape = (82790, 18)

df.columns =
['item_id', 'waist', 'size', 'quality', 'cup size', 'hips', 'bra size', 'category', 'bust', 'height', 'user_name', 'length', 'fit', 'user_id', 'shoe size', 'shoe width', 'review_summary', 'review_text']


/tmp/ipykernel_487/3614110208.py:1: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATASET_PATH)


,item_id,waist,size,quality,cup size,hips,bra size,category,bust,height,user_name,length,fit,user_id,shoe size,shoe width,review_summary,review_text
0,123373,29.0,7,5.0,d,38.0,34.0,new,36.0,5ft 6in,Emily,just right,small,991571,NaN,NaN,NaN,NaN
1,123373,31.0,13,3.0,b,30.0,36.0,new,NaN,5ft 2in,sydneybraden2001,just right,small,587883,NaN,NaN,NaN,NaN
2,123373,30.0,7,2.0,b,NaN,32.0,new,NaN,5ft 7in,Ugggh,slightly long,small,395665,9.0,NaN,NaN,NaN
3,123373,NaN,21,5.0,dd/e,NaN,NaN,new,NaN,NaN,alexmeyer626,just right,fit,875643,NaN,NaN,NaN,NaN
4,123373,NaN,18,5.0,b,NaN,36.0,new,NaN,5ft 2in,dberrones1,slightly long,small,944840,NaN,NaN,NaN,NaN



Valeurs manquantes par colonne :


,missing_count
waist,79908
bust,70936
shoe width,64183
shoe size,54875
hips,26726
review_summary,6732
review_text,6732
cup size,6255
bra size,6018
height,1107


## 9. Appel du script `src.training.train_fit_model`

Le notebook ne simule pas un entrainement : si `DATASET_PATH` est absent ou invalide, le script echouera explicitement.

In [20]:
EPOCHS = 20
BATCH_SIZE = 64

if not Path(DATASET_PATH).exists():
    raise FileNotFoundError(f'Dataset absent : {DATASET_PATH}')

os.chdir(REPO_DIR)
print(f'Repertoire courant : {Path.cwd()}')
!python -m src.training.train_fit_model --dataset "{DATASET_PATH}" --epochs {EPOCHS} --batch-size {BATCH_SIZE}

Repertoire courant : /content/fit-outfit-advisor
/usr/bin/python3: Error while finding module specification for 'src.training.train_fit_model' (ModuleNotFoundError: No module named 'src')


## 10. Metriques et artefacts generes

In [21]:
artifact_paths = [
    REPO_DIR / 'models' / 'fit_model.keras',
    REPO_DIR / 'models' / 'encoders' / 'fit_preprocessor.joblib',
    REPO_DIR / 'models' / 'encoders' / 'fit_label_encoder.joblib',
    REPO_DIR / 'models' / 'encoders' / 'fit_metadata.json',
]

print('Artefacts attendus :')
for path in artifact_paths:
    status = 'OK' if path.exists() else 'ABSENT'
    size = path.stat().st_size if path.exists() else 0
    print(f'{status:6} {size:>12} bytes  {path}')

missing_artifacts = [path for path in artifact_paths if not path.exists()]
if missing_artifacts:
    raise FileNotFoundError('Artefacts manquants apres entrainement : ' + ', '.join(map(str, missing_artifacts)))

metadata_path = REPO_DIR / 'models' / 'encoders' / 'fit_metadata.json'
print('\nMetadata fit :')
print(metadata_path.read_text(encoding='utf-8'))

Artefacts attendus :
ABSENT            0 bytes  /content/fit-outfit-advisor/models/fit_model.keras
ABSENT            0 bytes  /content/fit-outfit-advisor/models/encoders/fit_preprocessor.joblib
ABSENT            0 bytes  /content/fit-outfit-advisor/models/encoders/fit_label_encoder.joblib
ABSENT            0 bytes  /content/fit-outfit-advisor/models/encoders/fit_metadata.json


FileNotFoundError: Artefacts manquants apres entrainement : /content/fit-outfit-advisor/models/fit_model.keras, /content/fit-outfit-advisor/models/encoders/fit_preprocessor.joblib, /content/fit-outfit-advisor/models/encoders/fit_label_encoder.joblib, /content/fit-outfit-advisor/models/encoders/fit_metadata.json

## 11. Copie des artefacts finis vers Google Drive

In [ ]:
import shutil

DRIVE_ARTIFACT_DIR = Path('/content/drive/MyDrive/fit-outfit-advisor/artifacts/modcloth_fit')
DRIVE_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

for path in artifact_paths:
    destination = DRIVE_ARTIFACT_DIR / path.name
    shutil.copy2(path, destination)
    print(f'Copie : {path} -> {destination}')

print(f'\nArtefacts disponibles dans Google Drive : {DRIVE_ARTIFACT_DIR}')